# [Deep Mutual Learning](https://arxiv.org/pdf/1706.00384)

## Introduction

Knowledge Distillation is a training technique where a smaller or simpler student model is trained to mimic the output behavior of a larger or better-performing teacher model, with the goal of achieving similar performance more efficiently.

Deep Mutual Learning (DML) takes this concept and applies it to a cohort of student models, where **students are trained to mimic one another.** 

**Gap**: Standard distillation requires an already-trained, larger teacher model prior to training a more efficient model.

**Improvement**: Deep Mutual Learning (aka mutual distillation) removes the requirement of having a trained teacher model AND **consistently shows better performance than standard distillation.** 

## Result

Quantitatively shows:
1. A student trained from mutual distillation performs better than training the student independently.
2. A student trained from mutual distillation performs better than a student trained from standard distillation.
3. Ensembling the trained cohort resulting from mutual distillation shows further improvements.

## Approach

Say we have two student networks, $1$ and $2$, here is DML algorithm:
1. For a sample of the training data, get outputs $p_1$ and $p_2$.
2. Update parameters (take gradient step) for Network $1$ following loss $L_1$ below.
3. For same sample of training data, get outputs $p_1$ and $p_2$.
4. Update parameters (take gradient step) for Network $2$ following loss $L_2$ below.

$$L_1 = L_{C_1} + D_{KL}(p_2 ||  p_1)$$
$$L_2 = L_{C_2} + D_{KL}(p_1 || p_2)$$

Repeat till convergence.

Can be extended to more than 2 networks, K:

$$L_k = L_{C_k} + \frac{1}{K-1}\sum_{l=1, l \neq k}^{K}{D_{KL}(p_l || p_k)}$$

## Application

In [215]:
import torch
from torch import nn
import torch.nn.functional as F

class SmallNet(nn.Module):
    def __init__(self, input_size=(28,28), output_size=10, num_neurons=200):
        super(SmallNet, self).__init__()
        input_size = np.prod(input_size)
        self.fc1 = nn.Linear(input_size, num_neurons)
        self.fc2 = nn.Linear(num_neurons, num_neurons)
        self.out = nn.Linear(num_neurons, output_size)

    def forward(self, x):
        x = self.fc1(x)
        x = F.relu(x)
        
        x = self.fc2(x)
        x = F.relu(x)
        
        logits = self.out(x)
        return logits

In [216]:
# Data
from torchvision import datasets, transforms

# Jitter 2 pixels
jitter = 2 / 28
train_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.RandomAffine(degrees=0, translate=(jitter, jitter)),
])
test_transform = transforms.Compose([
    transforms.ToTensor(),
])

# Loading MNIST data
train_data = datasets.MNIST(root='data', train=True, download=True, transform=train_transform)
test_data = datasets.MNIST(root='data', train=False, download=True, transform=test_transform)

# Create data loaders
BATCH_SIZE = 128
train_loader = torch.utils.data.DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False)

train_data

Dataset MNIST
    Number of datapoints: 60000
    Root location: data
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
               RandomAffine(degrees=[0.0, 0.0], translate=(0.07142857142857142, 0.07142857142857142))
           )

In [254]:
from tqdm.notebook import tqdm

def train_model(model, optimizer, num_epochs=20, gamma=0.95):
    model = model.to(device)
    scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=gamma)
    for epoch in range(num_epochs):
        model.train()  # Set the model to training mode
        running_loss = 0.0
    
        progress_bar = tqdm(total=len(train_loader))
    
        for inputs, labels in train_loader:
            inputs, labels = inputs.view(inputs.shape[0], -1).to(device), labels.to(device)
            optimizer.zero_grad()  # Zero the gradients
            
            outputs = model(inputs)  # Forward pass
            loss = criterion(outputs, labels)  # Compute the loss
            loss.backward()  # Backward pass
            optimizer.step()  # Update the weights
            running_loss += loss.item() * inputs.size(0)
            
            progress_bar.update(1)
            progress_bar.set_description(f"Loss: {loss.item():.4f}")
        scheduler.step()

        next_lr = optimizer.param_groups[0]["lr"]
        all_preds, all_labels = test_model(model)
        num_instances = len(all_preds)
        correct = (all_preds == all_labels).sum()
        accuracy =  correct / num_instances
        errors = num_instances - correct
        epoch_loss = running_loss / len(train_loader.dataset)
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss:.4f}, Accuracy: {accuracy}, Errors: {errors}, next LR: {next_lr}")

def test_model(model):
    model.eval()
    
    all_preds = []
    all_labels = []

    # progress_bar = tqdm(total=len(test_loader))

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.view(inputs.shape[0], -1).to(device)
            labels = labels.to(device)

            outputs = model(inputs)
            preds = outputs.argmax(dim=-1)

            all_preds.append(preds.cpu())
            all_labels.append(labels.cpu())

            # progress_bar.update(1)

    all_preds = torch.cat(all_preds)
    all_labels = torch.cat(all_labels)

    return all_preds, all_labels

In [255]:
import numpy as np 

LR = 1e-1
MOMENTUM = 0.9
large_model = LargeNet()
optimizer = torch.optim.SGD(large_model.parameters(), lr=LR, momentum=MOMENTUM)
criterion = nn.CrossEntropyLoss()
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [256]:
LR = 1e-1
small_model = SmallNet()
optimizer = torch.optim.SGD(small_model.parameters(), lr=LR, momentum=MOMENTUM)
criterion = nn.CrossEntropyLoss()

In [257]:
train_model(small_model, optimizer, num_epochs=50)

  0%|          | 0/469 [00:00<?, ?it/s]

Epoch 1/50, Loss: 0.4823, Accuracy: 0.9678000211715698, Errors: 322, next LR: 0.095


  0%|          | 0/469 [00:00<?, ?it/s]

Epoch 2/50, Loss: 0.1634, Accuracy: 0.9726999998092651, Errors: 273, next LR: 0.09025


  0%|          | 0/469 [00:00<?, ?it/s]

...

Epoch 48/50, Loss: 0.0155, Accuracy: 0.9904000163078308, Errors: 96, next LR: 0.008525759033430816


  0%|          | 0/469 [00:00<?, ?it/s]

Epoch 49/50, Loss: 0.0153, Accuracy: 0.9898999929428101, Errors: 101, next LR: 0.008099471081759275


  0%|          | 0/469 [00:00<?, ?it/s]

Epoch 50/50, Loss: 0.0155, Accuracy: 0.9904000163078308, Errors: 96, next LR: 0.007694497527671311


In [250]:
models = [SmallNet() for _ in range(3)]
optimizers = []
for model in models:
    opt = torch.optim.SGD(model.parameters(), lr=LR, momentum=MOMENTUM)
    optimizers.append(opt)

In [251]:
from torch.utils.tensorboard import SummaryWriter

writer = SummaryWriter("runs/mutual_distillation")

In [252]:
def mutual_distil_model(models, optimizers, num_epochs=50, gamma=0.95, alpha=0.60, T=2):
    for model in models:
        model = model.to(device)

    schedulers = [
        torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=gamma)
        for optimizer in optimizers
    ]
        
    for epoch in range(num_epochs):
        for model in models:
            model.train()  # Set the model to training mode
            
        running_loss = 0.0
        progress_bar = tqdm(total=len(train_loader))
        curr_student_index = 0
        for batch, (inputs, labels) in enumerate(train_loader):
            inputs, labels = inputs.view(inputs.shape[0], -1).to(device), labels.to(device)

            global_step = epoch * len(train_loader) + batch

            for optimizer in optimizers:
                optimizer.zero_grad()  # Zero the gradients

            for curr_student_index, model in enumerate(models):
                # Forward passes
                logits = torch.stack([
                    model(inputs)
                    for model in models
                ])
    
                log_probs = F.log_softmax(logits / T, dim=-1)
                probs = F.softmax(logits / T, dim=-1)
            
                curr_student_log_probs = log_probs[curr_student_index]
                other_student_probs = torch.cat([
                    probs[:curr_student_index],
                    probs[curr_student_index + 1:]
                ], dim=0).detach()
                kl_divergences = F.kl_div(
                    curr_student_log_probs.unsqueeze(0),
                    other_student_probs,
                    reduction='none'
                ).sum(dim=-1)

                # Compute the loss
                mutual_distillation_loss = ((1 / kl_divergences.shape[0]) * kl_divergences.sum(dim=0)).mean() 
                mutual_distillation_loss *= T**2
                curr_student_ce_loss = criterion(logits[curr_student_index], labels)
                # print(alpha * curr_student_ce_loss, (1-alpha) * mutual_distillation_loss)
                loss = alpha * curr_student_ce_loss + (1-alpha) * mutual_distillation_loss

                writer.add_scalars(
                    f"loss/model{curr_student_index}",
                    {
                        "ce_loss": alpha * curr_student_ce_loss.item(),
                        "dml_loss": (1 - alpha) * mutual_distillation_loss.item(),
                        "total_loss": loss.item()
                    },
                    global_step
                )
                    
                loss.backward()  # Backward pass
                optimizers[curr_student_index].step()  # Update the weights
            
            running_loss += loss.item() * inputs.size(0)
            progress_bar.update(1)
            progress_bar.set_description(f"Loss: {loss.item():.4f}")
            curr_student_index = (curr_student_index + 1) % len(models)

        for scheduler in schedulers:
            scheduler.step()

        next_lr = optimizer.param_groups[0]["lr"]
        epoch_loss = running_loss / len(train_loader.dataset)
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss:.4f}, next LR: {next_lr}")
        for curr_student_index, model in enumerate(models):
            all_preds, all_labels = test_model(model) # change to average of all models rather than 1? 
            num_instances = len(all_preds)
            correct = (all_preds == all_labels).sum()
            accuracy = correct / num_instances
            errors = num_instances - correct
            writer.add_scalar(f"model{curr_student_index}/next_lr", next_lr, epoch)
            writer.add_scalar(f"model{curr_student_index}/accuracy", accuracy, epoch)
            writer.add_scalar(f"model{curr_student_index}/errors", errors, epoch)
            print(f"     Model: {curr_student_index}, Accuracy: {accuracy}, Errors: {errors}")

In [253]:
mutual_distil_model(models, optimizers)

  0%|          | 0/469 [00:00<?, ?it/s]

Epoch 1/50, Loss: 0.4474, next LR: 0.095
     Model: 0, Accuracy: 0.9487000107765198, Errors: 513
     Model: 1, Accuracy: 0.9491999745368958, Errors: 508
     Model: 2, Accuracy: 0.9491999745368958, Errors: 508


  0%|          | 0/469 [00:00<?, ?it/s]

Epoch 2/50, Loss: 0.1487, next LR: 0.09025
     Model: 0, Accuracy: 0.9677000045776367, Errors: 323
     Model: 1, Accuracy: 0.9675999879837036, Errors: 324
     Model: 2, Accuracy: 0.967199981212616, Errors: 328


  0%|          | 0/469 [00:00<?, ?it/s]

...

Epoch 48/50, Loss: 0.0250, next LR: 0.008525759033430816
     Model: 0, Accuracy: 0.9915000200271606, Errors: 85
     Model: 1, Accuracy: 0.9912999868392944, Errors: 87
     Model: 2, Accuracy: 0.9921000003814697, Errors: 79


  0%|          | 0/469 [00:00<?, ?it/s]

Epoch 49/50, Loss: 0.0247, next LR: 0.008099471081759275
     Model: 0, Accuracy: 0.991599977016449, Errors: 84
     Model: 1, Accuracy: 0.991599977016449, Errors: 84
     Model: 2, Accuracy: 0.9922999739646912, Errors: 77


  0%|          | 0/469 [00:00<?, ?it/s]

Epoch 50/50, Loss: 0.0251, next LR: 0.007694497527671311
     Model: 0, Accuracy: 0.9915000200271606, Errors: 85
     Model: 1, Accuracy: 0.9911999702453613, Errors: 88
     Model: 2, Accuracy: 0.9915000200271606, Errors: 85


In [258]:
def test_models(models):
    for model in models:
        model.eval()
    
    all_preds = []
    all_labels = []

    # progress_bar = tqdm(total=len(test_loader))

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.view(inputs.shape[0], -1).to(device)
            labels = labels.to(device)

            output = torch.stack([
                model(inputs) for model in models
            ]).mean(dim=0)
            
            preds = output.argmax(dim=-1)

            all_preds.append(preds.cpu())
            all_labels.append(labels.cpu())

            # progress_bar.update(1)

    all_preds = torch.cat(all_preds)
    all_labels = torch.cat(all_labels)

    return all_preds, all_labels

In [259]:
all_preds, all_labels = test_models(models)
num_instances = len(all_preds)
correct = (all_preds == all_labels).sum()
accuracy =  correct / num_instances
errors = num_instances - correct
print(f"Accuracy: {accuracy}, Errors: {errors}")

Accuracy: 0.9919000267982483, Errors: 81
